# TNG300：観測条件を通した環境情報と descendant mass の直接予測

**実行場所：IllustrisTNG公式JupyterLab。TNG300-1のgroup catalogとSubLinkを使用する。**
この1冊に読み込み・追跡・環境計算・推論の全関数を含む。追加のローカルPythonファイル、
09/10の実行済みkernelは不要。上から順に実行する。

研究の問いは、銀河の観測に対応する情報から $z=0$ のhost質量をどこまで予測できるか、
特に広い環境の情報がhalo massだけを使う予測へ何を加えるか、である。
ノートブック10の $M_\star$ と3D球内銀河数の比較を、以下の実験へ拡張する。

| 解析 | 目的 | 比較する情報 |
|---|---|---|
| 1. 模擬観測と完全性 | 3Dの環境を観測条件でどこまで回収できるか | 真の位置・質量、測光赤方偏移、分光、深い選択 |
| 2. 多スケール環境 | コアと外縁が異なる情報を持つか | 1,2,5,10,15 cMpcの銀河数・質量和・形状 |
| 3. Intermediate | 真の $M_z$ を与えても環境が予測に役立つか | $P(M_0\mid M_z,z)$ と $P(M_0\mid M_z,D,z)$ |
| 4. Finalと二段階法 | 同じ観測入力から直接予測する価値を調べる | $P(M_0\mid D,z)$ と $\int P(M_0\mid M_z,z)P(M_z\mid D,z)dM_z$ |
| 5. 領域の行先監査 | 銀河中心のラベルが領域全体を代表するか | 同一領域の銀河がどの $z=0$ hostへ配分されるか |

**対象とラベルを固定する。** 最初の実装は「観測で選んだanchor銀河」を単位とし、
ラベルはその銀河のSubLink descendantが属する $z=0$ FoFの $M_{200c}$。
各環境・追観測実験でanchorとラベルを変えない。対象は中心・衛星の両方で、
初期halo massや最終cluster massによる事前選択をしない。
銀河の密度ピークを探索・重複除去して作るprotocluster候補カタログとは異なる。
解析5は将来の「領域全体を入力する」モデルへ進むためのラベル監査であり、主解析のラベルは変更しない。

**観測モデルの範囲。** この実装は、誤差を加えたSED恒星質量、赤方偏移、質量依存の検出率、
分光抽出率による制御実験である。実際のJWST flux/EW選択、UVの放射輸送、
ALMAの線・塵flux、Lyαや中性ガスの尤度は含まない。SFR追加実験も瞬間SFRに誤差を加えたproxy。
これらを実観測の較正済み予測、あるいは観測時間あたりの最適化結果として解釈しない。

背景：[文献調査](../analysis_summary/highz_protocluster_observables_literature_ja.md)、
[Witten et al. 2026, §5.1](https://arxiv.org/html/2605.28930v1#S5.SS1)、
[Helton et al. 2024](https://arxiv.org/abs/2311.04270)、
[Baxter et al., updated v3](https://arxiv.org/abs/2504.03836v3)。
文献調査ファイルは、単独アップロード時にはなくても実行に影響しない。


## 実行設定と保存先

既定は `MODE='tng'`、snapshot 8（実際の $z\simeq8.012$）。まずこの時刻で全処理を実行する。
`Config(snapshots=(6,7,...,16))` とすれば複数時刻にも対応するが、各時刻に独立した予測器を学習し、
CVの平均損失は時刻ごとに等重みとする。時刻間の補間・lightconeの生成は行わない。

`BASE_PATH=None` は `TNG_OUTPUT_DIR` と公式JupyterLabの標準パスを探索する。
見つからなければ `'/home/tnguser/sims.TNG/TNG300-1/output'` など実際の場所を指定する。
必要なpackageは `numpy,pandas,scipy,scikit-learn,h5py,matplotlib,illustris_python`。
インストールを自動実行しない。

`MODE='synthetic_demo'` はソフトウェアの動作確認専用。銀河位置・質量・行先を人工的に作り、
すべての表と図にdemo表示を付ける。TNG解析の代替・科学的な結果ではない。

出力は `protohalo/cache/tng300_observational_environment/` 以下。リポジトリから実行すれば既存の
`protohalo` を検出し、単独アップロードなら作業ディレクトリの下に作成する。
設定・入力ファイルのstat・実装のhashごとに別のrunを作り、09/10の結果を上書きしない。


In [ ]:
"""Mock observing experiments and descendant-mass densities for notebook 11.

No simulation is read and no output is written on import. The observing model
is deliberately phenomenological: noisy SED masses, Gaussian redshift errors,
mass-dependent completeness, and spectroscopic sampling, not synthetic JWST
fluxes, radiative transfer, or a calibrated instrument selection function.
"""
from __future__ import annotations

from collections import OrderedDict
from dataclasses import asdict, dataclass
from itertools import product
from pathlib import Path
import hashlib
import json
import os
import sqlite3
import tempfile

import numpy as np
import pandas as pd
from scipy.optimize import brentq
from scipy.spatial import cKDTree
from scipy.special import expit, logsumexp, ndtr
from sklearn.ensemble import ExtraTreesRegressor

VERSION = 'observable-environment-v1'
C_KMS = 299792.458
POS = ['x_cMpc', 'y_cMpc', 'z_cMpc']
VEL = ['vx_kms', 'vy_kms', 'vz_kms']


@dataclass(frozen=True)
class Config:
    snapshots: tuple = (8,)
    seed: int = 42
    split_seed: int = 317
    archive_logmstar_min: float = 8.0
    anchor_logmstar_min: float = 9.0
    shallow_logmstar_min: float = 9.0
    deep_logmstar_min: float = 8.5
    mass_error_dex: float = 0.2
    sfr_error_dex: float = 0.3
    completeness_mid: float = 8.6
    deep_completeness_mid: float = 8.1
    completeness_width: float = 0.25
    photoz_sigma_over_1pz: float = 0.02
    spec_sigma_kms: float = 50.0
    spec_fraction: float = 0.5
    deep_spec_fraction: float = 0.8
    velocity_halfwidth_kms: float = 1200.0
    radii_cMpc: tuple = (1., 2., 5., 10., 15.)
    los_axis: int = 2
    test_fraction: float = 0.2
    cv_folds: int = 3
    max_anchors_per_snapshot: int = 2000
    cv_cap_per_snapshot: int = 150
    test_cap_per_snapshot: int = 300
    n_trees: int = 64
    max_depth: int = 10
    leaf_grid: tuple = (10, 20, 50)
    ywidth_grid: tuple = (0.15, 0.30)
    mwidth_grid: tuple = (0.10, 0.25)
    bridge_grid_size: int = 101
    min_train: int = 40
    min_validation: int = 8
    min_hosts: int = 10
    bootstrap_draws: int = 200
    region_audit_n: int = 30
    region_radius_cMpc: float = 10.0

    def validate(self, boxsize=None):
        if not self.snapshots or len(set(self.snapshots)) != len(self.snapshots):
            raise ValueError('Use distinct snapshots')
        if self.los_axis not in (0, 1, 2):
            raise ValueError('los_axis must be 0, 1 or 2')
        radii = np.asarray(self.radii_cMpc, float)
        if not len(radii) or not np.isfinite(radii).all() or np.any(radii <= 0) or np.any(np.diff(radii) <= 0):
            raise ValueError('Radii must be finite, positive, and strictly increasing')
        if self.deep_logmstar_min < self.archive_logmstar_min:
            raise ValueError('Deep selection must not extend below the stored truth floor')
        if self.deep_logmstar_min > self.shallow_logmstar_min:
            raise ValueError('Deep mass threshold must be at least as faint as shallow')
        if self.deep_completeness_mid > self.completeness_mid:
            raise ValueError('Deep imaging must have at least shallow completeness')
        if not 0 < self.spec_fraction <= self.deep_spec_fraction <= 1:
            raise ValueError('Invalid spectroscopic fractions')
        nonnegative = [self.mass_error_dex, self.sfr_error_dex,
                       self.photoz_sigma_over_1pz, self.spec_sigma_kms]
        if not np.isfinite(nonnegative).all() or min(nonnegative) < 0:
            raise ValueError('Errors must be finite and nonnegative')
        if self.completeness_width <= 0 or self.velocity_halfwidth_kms <= 0:
            raise ValueError('Positive completeness width and velocity window required')
        if not 0 < self.test_fraction < .5 or self.cv_folds < 2:
            raise ValueError('Invalid spatial split')
        if self.n_trees < 1 or min(self.leaf_grid) < 2 or min(self.ywidth_grid) <= 0 or min(self.mwidth_grid) <= 0:
            raise ValueError('Invalid density model parameters')
        if self.bridge_grid_size < 21 or self.bootstrap_draws < 1:
            raise ValueError('Increase integration grid / bootstrap draws')
        if min(self.min_train, self.min_validation, self.min_hosts,
               self.max_anchors_per_snapshot, self.cv_cap_per_snapshot,
               self.test_cap_per_snapshot) < 1:
            raise ValueError('Sample sizes must be positive')
        if boxsize is not None and 4*radii[-1] >= boxsize*(1-self.test_fraction):
            raise ValueError('Box too small for the transverse spatial embargo')


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()


def signature(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()


def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, name = tempfile.mkstemp(prefix=path.name+'.', suffix='.tmp', dir=str(path.parent))
    try:
        with os.fdopen(fd, 'w', encoding='utf-8', newline='') as stream:
            frame.to_csv(stream, index=False)
        Path(name).replace(path)
    finally:
        if Path(name).exists():
            Path(name).unlink()


def discover_base(explicit=None):
    value = explicit or os.environ.get('TNG_OUTPUT_DIR')
    if value:
        base = Path(value).expanduser().resolve()
        if not base.is_dir():
            raise FileNotFoundError(base)
        return base
    for root in [Path.cwd(), *Path.cwd().parents, Path('/home/tnguser')]:
        for name in ('TNG300-1', 'L205n2500TNG'):
            base = root/'sims.TNG'/name/'output'
            if base.is_dir():
                return base.resolve()
    raise FileNotFoundError('Set BASE_PATH or TNG_OUTPUT_DIR to TNG300-1/output')


def periodic_delta(a, b, box):
    return (np.asarray(a)-np.asarray(b)+box/2) % box-box/2


def rtag(radius):
    return ('%g' % radius).replace('.', 'p')


def validate_raw(frame, meta):
    required = set(POS+VEL+['SubhaloID', 'snapshot', 'redshift', 'logMstar',
                           'SFR', 'logMz', 'GroupID', 'Nstar'])
    if not required.issubset(frame) or frame.empty:
        raise ValueError('Incomplete raw galaxy table')
    if frame.duplicated(['snapshot', 'SubhaloID']).any():
        raise ValueError('Duplicate source galaxies')
    if frame.snapshot.nunique() != 1 or frame.redshift.nunique() != 1:
        raise ValueError('A raw table must contain exactly one snapshot')
    if not np.isfinite(frame[POS+VEL+['logMstar', 'SFR']].to_numpy(float)).all() or (frame.SFR < 0).any():
        raise ValueError('Nonfinite positions, velocities, stellar masses, or SFR')
    if (frame[POS].to_numpy() < 0).any() or (frame[POS].to_numpy() >= meta['boxsize_cMpc']).any():
        raise ValueError('Coordinates must lie inside the periodic box')


In [ ]:
from dataclasses import replace
from IPython.display import display
import matplotlib.pyplot as plt
import platform
import scipy
import sklearn

MODE = 'tng'  # 'synthetic_demo' is software verification only
BASE_PATH = None
# Optional: notebook-09 galaxies_all.csv with its manifest.json alongside.
# None means directly trace selected anchors; notebook 09 is not a prerequisite.
LEGACY_CSV = None
CFG = Config(snapshots=(8,))
IMPLEMENTATION_SHA256 = 'dde7934690f7e425a85e8e62f3cdc51370be054e648ebd4420a080b1bbb0e704'

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p/'protohalo'/'notebooks').is_dir()), cwd)
CACHE_ROOT = PROJECT_ROOT/'protohalo'/'cache'/'tng300_observational_environment'
if MODE not in ('tng', 'synthetic_demo'):
    raise ValueError('Unknown MODE')
DEMO = MODE == 'synthetic_demo'
LABEL_PREFIX = '[SYNTHETIC DEMO] ' if DEMO else 'TNG300-1 mock observing experiment: '
if DEMO:
    CFG = replace(CFG, n_trees=8, leaf_grid=(4,), ywidth_grid=(.2,), mwidth_grid=(.15,),
                  max_anchors_per_snapshot=400, cv_cap_per_snapshot=15, test_cap_per_snapshot=20,
                  min_train=15, min_validation=3, min_hosts=3, bootstrap_draws=30,
                  region_audit_n=5, bridge_grid_size=61)
CFG.validate()
print(LABEL_PREFIX+str(asdict(CFG)))
print('Cache:', CACHE_ROOT)


## 入力：物理的な銀河集団と、独立した子孫ラベル

**目的。** 周辺銀河の観測可能性と、子孫を追えるかどうかを混同しない。
tracerは同時刻の `SubhaloFlag=1`、真の $M_\star\geq10^8M_\odot$ の全保存銀河。
環境の計算にtracerの将来質量・追跡成功・train/test所属を使わない。
この保存下限は観測完全性でも分解能収束の保証でもない。

**手法。** 恒星質量は `SubhaloMassType[:,4]`、SFRは `SubhaloSFR`。
位置はckpc/hからcMpc、`SubhaloVel`は物理的なpeculiar velocity [km/s]。
anchorの真の $M_z$ は現在のFoFの `Group_M_Crit200`。最終ラベルも $M_{200c}$ で統一する。
中心・衛星を区別する列を保存し、最初は同じ銀河重み付き母集団で比較する。

子孫はSubLink `RootDescendantID` の終点を読み、snapshot・SubfindID・tree IDを照合する。
`not_in_tree` と $z=0$ 以前の正常な終端を記録し、I/O errorを未追跡に置き換えない。
追跡はSQLiteへ100天体ごとにcommitし、通常のInterrupt時にもfinallyで保存する。
元のtreeファイルを更新した場合は別cacheになる。
任意指定の09 CSVはmanifestと元ファイルのstatが一致する場合だけ再利用する。

出典：[TNG field specifications](https://www.tng-project.org/data/docs/specifications/)、
[SubLink documentation](https://www.tng-project.org/data/docs/scripts/#sublink)。


In [ ]:
def load_tng_snapshot(base, snapshot, cfg, cache_root):
    """Read full physical tracers, with no descendant or train/test restriction."""
    import glob
    import h5py
    import illustris_python as il
    from illustris_python.snapshot import snapPath
    head = il.groupcat.loadHeader(str(base), int(snapshot))
    h = float(head['HubbleParam'])
    if not np.isclose(h, .6774, atol=1e-4) or not np.isclose(head['BoxSize'], 205000):
        raise ValueError('Expected TNG300 cosmology and geometry')
    with h5py.File(snapPath(str(base), int(snapshot)), 'r') as stream:
        attrs = stream['Header'].attrs
        ndm = int(attrs['NumPart_Total'][1]) + (int(attrs.get('NumPart_Total_HighWord', [0]*6)[1]) << 32)
    if ndm != 2500**3:
        raise ValueError('Expected TNG300-1 resolution')
    z = float(head['Redshift'])
    if not 5 <= z <= 9.01:
        raise ValueError('This notebook is configured for 5 <= z <= 9 (snapshot 6 brackets z=9)')
    hz = 100*h*np.sqrt(float(head['Omega0'])*(1+z)**3+float(head['OmegaLambda']))
    paths = sorted(glob.glob(str(Path(il.groupcat.gcPath(str(base), int(snapshot))).parent/'*.hdf5')))
    if not paths:
        raise FileNotFoundError('No group catalog chunks')
    meta = dict(version=VERSION, simulation='TNG300-1', base=str(base), snapshot=int(snapshot),
                redshift=z, h=h, H_kms_Mpc=float(hz), boxsize_cMpc=float(head['BoxSize'])/(1000*h),
                archive_logmstar_min=cfg.archive_logmstar_min,
                source_files=[[p, Path(p).stat().st_size, Path(p).stat().st_mtime_ns] for p in paths])
    directory = Path(cache_root)/('raw_'+signature(meta)[:16])
    path = directory/'galaxies.csv'
    if path.exists():
        frame = pd.read_csv(path)
        validate_raw(frame, meta)
        return frame, meta
    fields = ['SubhaloFlag', 'SubhaloMassType', 'SubhaloPos', 'SubhaloVel',
              'SubhaloSFR', 'SubhaloLenType', 'SubhaloGrNr']
    sub = il.groupcat.loadSubhalos(str(base), int(snapshot), fields=fields)
    group = il.groupcat.loadHalos(str(base), int(snapshot), fields=['Group_M_Crit200', 'GroupFirstSub'])
    mass = np.asarray(sub['SubhaloMassType'][:, 4], float)*1e10/h
    keep = (np.asarray(sub['SubhaloFlag']) == 1) & np.isfinite(mass) & (mass >= 10**cfg.archive_logmstar_min)
    ids = np.flatnonzero(keep)
    gids = np.asarray(sub['SubhaloGrNr'][ids], np.int64)
    halo_mass = np.asarray(group['Group_M_Crit200'], float)*1e10/h
    if ((gids < 0) | (gids >= len(halo_mass))).any():
        raise ValueError('Invalid high-z group ID')
    mz = halo_mass[gids]
    frame = pd.DataFrame(dict(SubhaloID=ids, snapshot=int(snapshot), redshift=z,
                              GroupID=gids, logMstar=np.log10(mass[ids]),
                              SFR=np.asarray(sub['SubhaloSFR'][ids], float),
                              Nstar=np.asarray(sub['SubhaloLenType'][ids, 4], np.int64),
                              logMz=np.log10(np.where(mz > 0, mz, np.nan)),
                              is_central=np.asarray(group['GroupFirstSub'])[gids] == ids))
    for j, name in enumerate(POS):
        frame[name] = np.asarray(sub['SubhaloPos'][ids, j], float)/(1000*h) % meta['boxsize_cMpc']
    # SubhaloVel is the physical peculiar velocity in km/s (no sqrt(a) factor).
    for j, name in enumerate(VEL):
        frame[name] = np.asarray(sub['SubhaloVel'][ids, j], float)
    validate_raw(frame, meta)
    directory.mkdir(parents=True, exist_ok=True)
    atomic_csv(frame, path)
    (directory/'manifest.json').write_text(json.dumps(meta, indent=2))
    return frame, meta


class DescendantStore:
    """Resumable endpoint tracking; no I/O exception is called 'untracked'."""
    def __init__(self, base, cache_root, legacy_csv=None):
        import glob
        import illustris_python as il
        self.il, self.base = il, str(base)
        paths = glob.glob(il.sublink.treePath(self.base, 'SubLink', '*'))
        for sn in (0, 99):
            paths.extend(glob.glob(str(Path(il.groupcat.gcPath(self.base, sn)).parent/'*.hdf5')))
        # Per-snapshot offsets are catalog dependencies of treeOffsets.
        for sn in range(100):
            p = Path(il.groupcat.offsetPath(self.base, sn))
            if p.exists():
                paths.append(str(p))
        if not paths:
            raise FileNotFoundError('Missing SubLink / group catalog files')
        self.fingerprint = dict(version=VERSION, base=self.base,
            target='z0 FoF Group_M_Crit200 of SubLink RootDescendantID',
            files=[[p, Path(p).stat().st_size, Path(p).stat().st_mtime_ns] for p in sorted(set(paths))])
        directory = Path(cache_root)/('traces_'+signature(self.fingerprint)[:16])
        directory.mkdir(parents=True, exist_ok=True)
        (directory/'manifest.json').write_text(json.dumps(self.fingerprint, indent=2))
        self.conn = sqlite3.connect(str(directory/'traces.sqlite'))
        self.conn.execute('CREATE TABLE IF NOT EXISTS traces (snap INTEGER, sid INTEGER, payload TEXT, PRIMARY KEY(snap,sid))')
        self.handles, self.memory, self.pending = OrderedDict(), {}, 0
        self.offsets = np.asarray(il.sublink.subLinkOffsets(self.base, 'SubLink'), np.int64)
        self.g0 = np.asarray(il.groupcat.loadSubhalos(self.base, 99, fields='SubhaloGrNr'), np.int64)
        h = float(il.groupcat.loadHeader(self.base, 99)['HubbleParam'])
        self.m0 = np.asarray(il.groupcat.loadHalos(self.base, 99, fields='Group_M_Crit200'), float)*1e10/h
        if legacy_csv is not None:
            # Existing 09 data are only reused after provenance and source-stat validation.
            legacy_csv = Path(legacy_csv)
            meta = json.loads(legacy_csv.with_name('manifest.json').read_text())
            if meta.get('version') != 'latent-host-v1' or Path(meta['base']).resolve() != Path(base).resolve():
                raise ValueError('Legacy catalog provenance differs from notebook 09')
            for p, size, mtime in meta['files']:
                stat = Path(p).stat()
                if (stat.st_size, stat.st_mtime_ns) != (size, mtime):
                    raise ValueError('Legacy catalog source files changed')
            old = pd.read_csv(legacy_csv)
            if old.duplicated(['snapshot', 'SubhaloID']).any():
                raise ValueError('Duplicate legacy trace keys')
            cols = ['TrackingStatus', 'SubhaloID_z0', 'TerminalSnap', 'GroupID_z0', 'logM0']
            for row in old.itertuples(index=False):
                payload = {k: getattr(row, k) for k in cols}
                self.conn.execute('INSERT OR IGNORE INTO traces VALUES (?,?,?)',
                                  (int(row.snapshot), int(row.SubhaloID), json.dumps(payload)))
            self.conn.commit()

    def trace(self, snapshot, sid):
        key = (int(snapshot), int(sid))
        if key in self.memory:
            return dict(self.memory[key])
        saved = self.conn.execute('SELECT payload FROM traces WHERE snap=? AND sid=?', key).fetchone()
        if saved:
            payload = json.loads(saved[0])
        else:
            import h5py
            row, _, tree_id = map(int, self.il.sublink.treeOffsets(self.base, *key, 'SubLink'))
            payload = dict(TrackingStatus='not_in_tree', SubhaloID_z0=-1, TerminalSnap=-1,
                           GroupID_z0=-1, logM0=np.nan)
            if row >= 0:
                chunk = int(np.searchsorted(self.offsets, row, side='right')-1)
                if chunk < 0:
                    raise ValueError('Invalid SubLink file offset')
                if chunk not in self.handles:
                    self.handles[chunk] = h5py.File(self.il.sublink.treePath(self.base, 'SubLink', chunk), 'r')
                    if len(self.handles) > 4:
                        self.handles.popitem(last=False)[1].close()
                stream = self.handles[chunk]
                local = row-int(self.offsets[chunk])
                if not 0 <= local < len(stream['SubhaloID']):
                    raise ValueError('Seed outside tree file')
                if (int(stream['SubhaloID'][local]), int(stream['SnapNum'][local]), int(stream['SubfindID'][local])) != (tree_id, *key):
                    raise ValueError('SubLink seed mismatch')
                root = int(stream['RootDescendantID'][local])
                endpoint = local+root-tree_id
                if not 0 <= endpoint < len(stream['SubhaloID']) or int(stream['SubhaloID'][endpoint]) != root:
                    raise ValueError('SubLink endpoint mismatch')
                end_snap = int(stream['SnapNum'][endpoint])
                if not key[0] <= end_snap <= 99:
                    raise ValueError('Invalid terminal snapshot')
                payload['TerminalSnap'] = end_snap
                payload['TrackingStatus'] = 'terminated_before_z0'
                if end_snap == 99:
                    s0 = int(stream['SubfindID'][endpoint])
                    if not 0 <= s0 < len(self.g0):
                        raise ValueError('Invalid z0 subhalo ID')
                    g0 = int(self.g0[s0])
                    if not 0 <= g0 < len(self.m0):
                        raise ValueError('Invalid z0 FoF ID')
                    mass = self.m0[g0]
                    payload.update(SubhaloID_z0=s0, GroupID_z0=g0,
                                   logM0=float(np.log10(mass)) if mass > 0 else np.nan,
                                   TrackingStatus='success' if np.isfinite(mass) and mass > 0 else 'invalid_host_mass')
            self.conn.execute('INSERT INTO traces VALUES (?,?,?)', (*key, json.dumps(payload)))
            self.pending += 1
            if self.pending >= 100:
                self.conn.commit()
                self.pending = 0
        self.memory[key] = payload
        return dict(payload)

    def close(self):
        self.conn.commit()
        self.conn.close()
        for stream in self.handles.values():
            stream.close()


## 解析1：同じ銀河集団を、浅い／深い観測と測光／分光へ写す

**目的。** 環境が持つ理想的な情報のうち、赤方偏移誤差・検出の不完全性・分光抽出で失われる情報を調べる。

**手法。** 各銀河に恒星質量のGaussian誤差（既定0.2 dex）を加える。
推定恒星質量に依存するlogistic検出率を用い、浅い／深い観測で同じ一様乱数を共有する。
分光tracerも同じ乱数で抽出し、deepをshallowの上位集合にする。
既定のtracer閾値は浅い観測が $\log M_{\star,obs}\geq9$、深い観測が8.5。
深い実験では分光抽出率も0.5から0.8へ増やすので、両方が変わる合同実験である。

anchorは浅い撮像で検出され、$\log M_{\star,obs}\geq9$ を満たす銀河から、ラベルを見る前に無作為抽出する。
最大2000天体/時刻のcapは計算量の制御で、質量や環境による上位選択ではない。
**すべてのanchorには別途systemic分光がある**と仮定し、誤差は既定50 km/s。
周辺tracerの測光誤差は $\sigma_z/(1+z)=0.02$、分光誤差は50 km/sとする。
catastrophic photo-z、flux依存の線検出、誤差間の相関は未実装。

SFR追加実験の量は $\log_{10}(\mathrm{SFR}+0.01)$ に0.3 dexの誤差を加えたproxy。
0.01はゼロSFRを有限値へ写す固定のfloorであり、観測上限値の尤度ではない。


In [ ]:
def mock_observe(raw, meta, cfg):
    """Common random numbers across shallow/deep and photo/spec experiments."""
    validate_raw(raw, meta)
    d = raw.sort_values('SubhaloID').reset_index(drop=True).copy()
    rng = np.random.default_rng(np.random.SeedSequence([cfg.seed, int(d.snapshot.iloc[0])]))
    n = len(d)
    d['logMstar_obs'] = d.logMstar+cfg.mass_error_dex*rng.normal(size=n)
    d['logSFR_proxy'] = np.log10(d.SFR+.01)+cfg.sfr_error_dex*rng.normal(size=n)
    udet, uspec, ez = rng.random(n), rng.random(n), rng.normal(size=n)
    d['p_detect_shallow'] = expit((d.logMstar_obs-cfg.completeness_mid)/cfg.completeness_width)
    d['p_detect_deep'] = expit((d.logMstar_obs-cfg.deep_completeness_mid)/cfg.completeness_width)
    d['detected_shallow'] = udet < d.p_detect_shallow
    d['detected_deep'] = udet < d.p_detect_deep
    d['spec_shallow'] = uspec < cfg.spec_fraction
    d['spec_deep'] = uspec < cfg.deep_spec_fraction
    scale = meta['H_kms_Mpc']/(1+meta['redshift'])
    d['v_systemic_true'] = scale*d[POS[cfg.los_axis]]+d[VEL[cfg.los_axis]]
    d['v_spec'] = d.v_systemic_true+cfg.spec_sigma_kms*ez
    d['v_photo'] = d.v_systemic_true+C_KMS*cfg.photoz_sigma_over_1pz*ez
    # Every selected anchor is assumed to have separately targeted spectroscopy.
    # spec_shallow/deep refer to the environmental tracers only.
    return d


def select_anchors(observed, cfg):
    selected = observed.loc[observed.detected_shallow & observed.logMstar_obs.ge(cfg.anchor_logmstar_min)].copy()
    if len(selected) > cfg.max_anchors_per_snapshot:
        selected = selected.sample(cfg.max_anchors_per_snapshot,
                                   random_state=cfg.seed+int(observed.snapshot.iloc[0]))
    return selected.sort_values('SubhaloID').reset_index(drop=True)


def attach_labels(anchors, store):
    rows = [dict(snapshot=int(r.snapshot), SubhaloID=int(r.SubhaloID),
                 **store.trace(int(r.snapshot), int(r.SubhaloID))) for r in anchors.itertuples(index=False)]
    if not rows:
        raise ValueError('No observed anchors; revise selection without using descendant mass')
    return anchors.merge(pd.DataFrame(rows), on=['snapshot', 'SubhaloID'], validate='one_to_one')


def wrapped_window_probability(delta, sigma, halfwidth, period):
    """Gaussian-error membership in a periodic velocity interval, flat prior.

    This is a phenomenological redshift-space membership weight. It is NOT
    probability of a shared z0 descendant, or a density-prior-calibrated p(z).
    """
    delta = periodic_delta(delta, 0., period)
    if not 0 < halfwidth < period/2 or sigma < 0 or not np.isfinite(sigma):
        raise ValueError('Invalid velocity window, uncertainty, or periodic extent')
    if sigma == 0:
        return (np.abs(delta) < halfwidth).astype(float)
    kmax = int(np.ceil(8*sigma/period))+1
    value = np.zeros_like(delta, float)
    for k in range(-kmax, kmax+1):
        value += ndtr((halfwidth-delta+k*period)/sigma)-ndtr((-halfwidth-delta+k*period)/sigma)
    return np.clip(value, 0., 1.)


## 解析2：多スケールの銀河数、質量分布、空間形状

**目的。** 中心の情報を、外縁・周辺の銀河の情報から分けて評価する。
半径は1,2,5,10,15 cMpcを事前固定する。testの相関を見て最適半径を選ばない。

**3D参照。** 真の位置の $r<R$ 球。`SubhaloID`で自分だけを除き、同じFoFの他銀河を含む。
**観測に対応する環境。** 視線を既定z軸に固定し、$R_\perp<R$ の全LOSを検索する。
赤方偏移空間の座標は $u=H(z)x_\parallel/(1+z)+v_{pec}$ に測定誤差を加えたもの。
分光環境では $|\Delta u|<1200$ km/sのtracerを数える。
測光環境ではGaussian誤差の下で、この窓に入る確率を重みにする：

$$w_i=\Phi[(v_{max}-\Delta u_i)/\sigma_{pair}]-\Phi[(-v_{max}-\Delta u_i)/\sigma_{pair}],
\qquad N_{eff}(R)=\sum_{R_{\perp,i}<R}w_i.$$

箱は周期境界なので、Gaussianの周期像の寄与も足す。測光のLOSを10/15 cMpcで事前切断しない。
これは平坦な赤方偏移priorでの模擬membershipであり、局所密度priorや実測 $p(z)$ で較正した値ではない。
宇宙膨張とpeculiar velocityを含むため、3D球と円柱は同じ体積・母集団ではない。
snapshotを周期的に使う実験であり、lightcone上の進化やbox外の独立構造は含まない。

保存する量は $\ln(1+N_{eff})$、$\log_{10}(1+\sum_i w_iM_{\star,obs,i})$、
最近傍距離、anchorと最も重い近傍のmass gap、重み付き2D位置共分散の楕円率。
形状は有効近傍数が3未満なら0とし、`has_shape`を併記する。近傍なしも専用flagを持つ。
proxy SFRの平均には`has_sfr_neighbors`を付ける。真の／推定されたhalo massや子孫情報を
これらの観測特徴量へ混ぜない。観測数は逆完全性補正せず、その観測条件のまま学習する。

$z\simeq8$で10 cMpcは半径約3.8分角。既定の15 cMpcまでの円形視野は
広域mosaicを前提とする理想化した一様視野で、実際のJWST footprintやmaskはまだ含まない。


In [ ]:
def environment_features(observed, anchors, meta, cfg, scenario):
    """Self excluded by ID; full periodic LOS queried for photo-z weights."""
    box, z = meta['boxsize_cMpc'], meta['redshift']
    rmax = max(cfg.radii_cMpc)
    truth = scenario == 'truth3d'
    if truth:
        use = observed.logMstar.gt(cfg.archive_logmstar_min)
    elif scenario in ('photo', 'spec'):
        use = observed.detected_shallow & observed.logMstar_obs.ge(cfg.shallow_logmstar_min)
        if scenario == 'spec':
            use &= observed.spec_shallow
    elif scenario == 'deep_spec':
        use = observed.detected_deep & observed.logMstar_obs.ge(cfg.deep_logmstar_min) & observed.spec_deep
    else:
        raise ValueError('Unknown observing scenario')
    tracers = observed.loc[use].reset_index(drop=True)
    perp = [a for a in range(3) if a != cfg.los_axis]
    xycols = [POS[a] for a in perp]
    positions = tracers[POS if truth else xycols].to_numpy(float)
    query = anchors[POS if truth else xycols].to_numpy(float)
    tree = cKDTree(positions, boxsize=box)
    nearby = tree.query_ball_point(query, rmax)
    lookup = observed.set_index('SubhaloID', verify_integrity=True)
    period = meta['H_kms_Mpc']*box/(1+z)
    if cfg.velocity_halfwidth_kms >= period/2:
        raise ValueError('Velocity window exceeds half the periodic LOS')
    rows = []
    for anchor, candidates in zip(anchors.itertuples(index=False), nearby):
        t = tracers.iloc[candidates]
        t = t.loc[t.SubhaloID.ne(anchor.SubhaloID)]
        delta = periodic_delta(t[POS if truth else xycols].to_numpy(),
                               [getattr(anchor, c) for c in POS if truth] if truth else
                               [getattr(anchor, c) for c in xycols], box)
        radius = np.linalg.norm(delta, axis=1)
        if truth:
            weight = np.ones(len(t))
        elif scenario == 'photo':
            sigma = np.hypot(C_KMS*cfg.photoz_sigma_over_1pz, cfg.spec_sigma_kms)
            weight = wrapped_window_probability(t.v_photo.to_numpy()-anchor.v_spec,
                                                sigma, cfg.velocity_halfwidth_kms, period)
        else:
            dv = periodic_delta(t.v_spec.to_numpy(), anchor.v_spec, period)
            weight = (np.abs(dv) < cfg.velocity_halfwidth_kms).astype(float)
        row = dict(snapshot=int(anchor.snapshot), SubhaloID=int(anchor.SubhaloID))
        mstar = t.logMstar.to_numpy() if truth else t.logMstar_obs.to_numpy()
        for r in cfg.radii_cMpc:
            inside = radius < r
            w = weight*inside
            tag = rtag(r)
            row['lnN_R'+tag] = float(np.log1p(w.sum()))
            row['log1pMstarSum_R'+tag] = float(np.log10(1+np.dot(w, 10**mstar)))
        within = (radius < rmax) & (weight > .5)
        row['has_neighbor'] = int(within.any())
        row['nearest_cMpc'] = float(radius[within].min()) if within.any() else float(rmax)
        anchor_mass = lookup.loc[anchor.SubhaloID, 'logMstar' if truth else 'logMstar_obs']
        row['mass_gap'] = float(anchor_mass-mstar[within].max()) if within.any() else 0.
        w = weight*(radius < rmax)
        row['mean_logSFR_proxy'] = float(np.dot(w, t.logSFR_proxy)/w.sum()) if w.sum() else 0.
        row['has_sfr_neighbors'] = int(w.sum() > 0)
        xy = periodic_delta(t[xycols].to_numpy(), [getattr(anchor, c) for c in xycols], box)
        neff = w.sum()**2/np.dot(w, w) if np.dot(w, w) else 0.
        row['has_shape'] = int(neff >= 3)
        row['ellipticity'] = 0.
        if neff >= 3:
            center = np.average(xy, weights=w, axis=0)
            cov = ((xy-center)*w[:, None]).T@(xy-center)/w.sum()
            eig = np.linalg.eigvalsh(cov)
            row['ellipticity'] = float((eig[-1]-eig[0])/eig.sum()) if eig.sum() else 0.
        rows.append(row)
    return pd.DataFrame(rows).add_prefix(scenario+'__').rename(columns={
        scenario+'__snapshot': 'snapshot', scenario+'__SubhaloID': 'SubhaloID'})


def feature_sets(cfg):
    counts = lambda s, radii: [s+'__lnN_R'+rtag(r) for r in radii]
    core = ['logMstar_obs']
    spec = core+counts('spec', cfg.radii_cMpc)
    geometry = spec+['spec__log1pMstarSum_R'+rtag(max(cfg.radii_cMpc)),
                     'spec__nearest_cMpc', 'spec__mass_gap', 'spec__has_neighbor',
                     'spec__ellipticity', 'spec__has_shape']
    population = geometry+['logSFR_proxy', 'spec__mean_logSFR_proxy', 'spec__has_sfr_neighbors']
    narrow = [r for r in cfg.radii_cMpc if r <= 5]
    if not narrow:
        narrow = [min(cfg.radii_cMpc)]
    return OrderedDict([
        ('halo_oracle', ['logMz']),
        ('halo_plus_environment', ['logMz']+counts('spec', cfg.radii_cMpc)),
        ('core', core), ('photo_environment', core+counts('photo', cfg.radii_cMpc)),
        ('spec_narrow', core+counts('spec', narrow)), ('spec_wide', spec),
        ('deep_spec', core+counts('deep_spec', cfg.radii_cMpc)),
        ('geometry', geometry), ('population_proxy', population),
        ('truth3d_reference', ['logMstar']+counts('truth3d', cfg.radii_cMpc)),
        ('two_stage_population', population),
    ])


## 評価設計：空間と最終hostを分離する

**目的。** 同じ最終hostや、重なった周辺領域がtrainとtestに入り、予測性能を高く見せることを避ける。

**手法。** LOSと直交する軸に、周期箱の20%幅のtest slabを一つ固定する。
境界の両側に $2R_{max}=30$ cMpcのembargoを置き、testと同じ $z=0$ hostへ行くtraining銀河も除く。
直交軸で分けるため、測光環境のLOSが長くてもtrain/testの円柱は重ならない。
残ったtraining slabを3分割し、CVでも同じ空間embargoと最終host purgeを行う。
複数時刻の場合は全時刻を通してhostをpurgeする。

既定の最小学習数は40、validationは8、学習側の最終host数は10。
これらは計算を成立させるための下限で、統計的精度を保証する数ではない。
候補leaf sizeは10/20/50とし、最も少ないCV学習標本でも2つのleafを作れる候補だけを
**全fold・全モデルに共通の規則で**使用する。例えば最小foldが53天体ならleaf=10/20を使い、
leaf=50は除外する。fold自体を抜いたり、空間embargoやhost purgeを緩めたりしない。

候補bandwidthや木のleaf sizeはtraining CVだけで選ぶ。CVが全foldで成立しない時刻は、
全モデルで共通に除外し、理由と天体数を保存する。疎な時刻へ無検証でパラメータを移さない。
testは一度固定して、全モデルを同じ天体上で採点する。
このembargoでも大スケールの相関・宇宙分散がなくなるわけではない。


In [ ]:
def spatial_split(frame, meta, cfg):
    """One global test slab; transverse 2*Rmax embargo and z0-host purging.

    Since the split axis is perpendicular to the LOS, even the full-depth
    photo-z cylinders of training and test anchors cannot overlap. Tracers
    themselves are observed data, and are never filtered by future labels.
    """
    box = meta['boxsize_cMpc']
    cfg.validate(box)
    axis = next(a for a in range(3) if a != cfg.los_axis)
    origin = np.random.default_rng(cfg.split_seed).uniform(0, box)
    d = frame.copy()
    d['split_coordinate'] = (d[POS[axis]]-origin) % box
    edge, buffer = cfg.test_fraction*box, 2*max(cfg.radii_cMpc)
    test = d.split_coordinate.lt(edge)
    train = d.split_coordinate.ge(edge+buffer) & d.split_coordinate.lt(box-buffer)
    test_hosts = set(d.loc[test, 'GroupID_z0'])
    host_purge = train & d.GroupID_z0.isin(test_hosts)
    d['partition'] = np.where(test, 'test', np.where(train & ~host_purge, 'train', 'embargo'))
    d.loc[host_purge, 'partition'] = 'host_purged'
    d['cv_fold'] = -1
    span = box-buffer-(edge+buffer)
    fold = np.floor((d.split_coordinate-(edge+buffer))/span*cfg.cv_folds).astype(int)
    d.loc[d.partition.eq('train'), 'cv_fold'] = fold.clip(0, cfg.cv_folds-1)
    if not set(d.loc[d.partition.eq('train'), 'GroupID_z0']).isdisjoint(test_hosts):
        raise AssertionError('Final host leakage')
    spec = dict(axis=axis, origin_cMpc=float(origin), boxsize_cMpc=float(box),
                test_edge_cMpc=float(edge), embargo_cMpc=float(buffer),
                cv_lower_cMpc=float(edge+buffer), cv_upper_cMpc=float(box-buffer))
    return d, spec


def cv_splits(train, split_spec, cfg):
    """Validation slabs are shared by all models, with spatial/host purging."""
    width = (split_spec['cv_upper_cMpc']-split_spec['cv_lower_cMpc'])/cfg.cv_folds
    splits, audit = [], []
    for fold in range(cfg.cv_folds):
        lo = split_spec['cv_lower_cMpc']+fold*width
        hi = lo+width
        val = train.cv_fold.eq(fold)
        # Linear distance is sufficient inside the already-embargoed train slab.
        x = train.split_coordinate
        far = (x < lo-split_spec['embargo_cMpc']) | (x >= hi+split_spec['embargo_cMpc'])
        held_hosts = set(train.loc[val, 'GroupID_z0'])
        fit = ~val & far & ~train.GroupID_z0.isin(held_hosts)
        a, b = train.loc[fit].copy(), train.loc[val].copy()
        if not set(a.GroupID_z0).isdisjoint(b.GroupID_z0):
            raise AssertionError('CV final-host leakage')
        splits.append((a, b))
        for sn in sorted(train.snapshot.unique()):
            aa, bb = a.loc[a.snapshot.eq(sn)], b.loc[b.snapshot.eq(sn)]
            audit.append(dict(fold=fold, snapshot=int(sn), N_train=len(aa), N_validation=len(bb),
                              hosts_train=aa.GroupID_z0.nunique(), hosts_validation=bb.GroupID_z0.nunique(),
                              **cv_requirements(len(aa), len(bb), aa.GroupID_z0.nunique(), cfg)))
    audit = pd.DataFrame(audit)
    usable = audit.groupby('snapshot').eligible.all()
    epochs = usable.index[usable].tolist()
    audit['used_by_all_models'] = audit.snapshot.isin(epochs)
    return splits, audit, epochs


def cv_requirements(n_train, n_validation, hosts_train, cfg):
    """Sample-size gate, independent of targets, scores, and held-out test.

    In addition to the explicit floor, at least the smallest candidate must
    have enough rows for two leaves. Larger leaves are checked separately
    against the smallest training fold shared by all selected epochs.
    """
    required = max(cfg.min_train, 2*min(cfg.leaf_grid))
    failures = []
    if n_train < required:
        failures.append('N_train=%d < required=%d' % (n_train, required))
    if n_validation < cfg.min_validation:
        failures.append('N_validation=%d < required=%d' % (n_validation, cfg.min_validation))
    if hosts_train < cfg.min_hosts:
        failures.append('hosts_train=%d < required=%d' % (hosts_train, cfg.min_hosts))
    return dict(eligible=not failures, required_train=int(required),
                max_splittable_leaf=int(n_train//2), reason='; '.join(failures) if failures else 'included')


## 条件付き確率分布の推定

**目的。** 中央値だけでなく、幅・非対称性・clusterになる確率を比較する。
高次元の環境特徴量を扱うため、10のproduct KDEからleaf分布を使うdensity estimatorへ変更する。

`ExtraTreesRegressor`をbootstrapなしで学習し、queryと同じleafに入ったtraining銀河へ
leaf内等重みを与え、全treeで平均する。応答は $y=\log_{10}(M_0/M_\odot)$。

$$\hat p(y\mid x)=\sum_j w_j(x)\,\mathcal N(y;y_j,h_y^2),\quad
w_j(x)=\frac1T\sum_t\frac{\mathbf1[l_t(x)=l_t(x_j)]}{n_{t,l_t(x)}}.$$

Gaussian混合は全実数上で正規化され、密度の単位はdex$^{-1}$。
標本数による候補判定を先に行い、その後leaf sizeと出力幅 $h_y$ をCVのNLLで選ぶ。
標本数より大きいleafを指定して、すべてのqueryが同じ分布になる候補を比較に混ぜない。
平滑化幅は観測誤差やモデル不確実性そのものではない。
これはシミュレーションの母集団に基づく経験的な条件付き分布であり、木のパラメータのBayesian posteriorではない。

**二段階法。** 同じ観測特徴量から $p(m_z\mid D)$ を学び、別の $p(y\mid m_z)$ と積分する。
$m_z=\log_{10}M_z$ のGaussian成分を101個のbinに解析的に積分し、各bin中心で第2段階を評価する。
点推定 $\hat M_z$ の代入にはしない。第1・第2段階の幅も、最終 $y$ のCV NLLで同時に選ぶ。
grid外の確率の切り落としを検査し、trainingの $M_z$ 範囲外へ積分された確率も保存する。
有限grid近似の感度は `bridge_grid_size` を変えたtraining側の再実験で確認できる。


In [ ]:
class GaussianMixture:
    """Normalized conditional density per dex in log10(M/Msun)."""
    def __init__(self, means, weights, width, hosts=None):
        self.means, self.weights = np.asarray(means, float), np.asarray(weights, float)
        self.width = float(width)
        if (self.means.ndim != 1 or self.means.shape != self.weights.shape or not len(self.means)
                or not np.isfinite(self.means).all() or not np.isfinite(self.weights).all()
                or (self.weights < 0).any() or self.weights.sum() <= 0
                or not np.isfinite(width) or width <= 0):
            raise ValueError('Invalid Gaussian mixture')
        self.weights = self.weights/self.weights.sum()
        self.hosts = np.arange(len(self.means)) if hosts is None else np.asarray(hosts)
        if len(self.hosts) != len(self.means):
            raise ValueError('Host length mismatch')

    def cdf(self, value):
        x = np.asarray(value, float)
        result = ndtr((x[..., None]-self.means)/self.width)@self.weights
        return float(result) if x.ndim == 0 else result

    def logpdf(self, value):
        with np.errstate(divide='ignore'):
            lw = np.log(self.weights)
        x = np.asarray(value, float)
        result = logsumexp(lw-.5*((x[..., None]-self.means)/self.width)**2, axis=-1)
        result -= np.log(self.width)+.5*np.log(2*np.pi)
        return float(result) if x.ndim == 0 else result

    def quantile(self, q):
        if not 0 < q < 1:
            raise ValueError('Use 0 < q < 1')
        return brentq(lambda y: self.cdf(y)-q, self.means.min()-12*self.width,
                      self.means.max()+12*self.width)

    def diagnostics(self):
        w = pd.Series(self.weights).groupby(self.hosts).sum().to_numpy()
        return dict(ESS_galaxy=float(1/np.dot(self.weights, self.weights)),
                    ESS_host=float(1/np.dot(w, w)))


class ForestDensity:
    """Leaf empirical response distributions, smoothed by Gaussian kernels.

    Every tree is fitted without bootstrap. A response gets 1/n_leaf weight
    in each query leaf, averaged across trees. This is an empirical density
    estimator, not a Bayesian posterior over tree parameters.
    """
    def __init__(self, cols, leaf, ywidth, cfg, target='logM0'):
        self.cols, self.width, self.target = list(cols), float(ywidth), target
        self.leaf = int(leaf)
        self.forest = ExtraTreesRegressor(n_estimators=cfg.n_trees, min_samples_leaf=int(leaf),
            max_depth=cfg.max_depth, max_features=1., bootstrap=False, n_jobs=1, random_state=cfg.seed)

    def fit(self, frame):
        x = frame[self.cols].to_numpy(float)
        y = frame[self.target].to_numpy(float)
        if not len(frame) or not np.isfinite(x).all() or not np.isfinite(y).all():
            raise ValueError('Training features and response must be finite')
        if len(frame) < 2*self.leaf:
            raise ValueError('N_train=%d cannot support two leaves of size %d; inspect cv_candidate_audit.csv'
                             % (len(frame), self.leaf))
        self.means, self.hosts = y, frame.GroupID_z0.to_numpy()
        self.minimum, self.maximum = x.min(axis=0), x.max(axis=0)
        self.forest.fit(x, y)
        leaves = self.forest.apply(x)
        self.lookup = []
        for j in range(leaves.shape[1]):
            column = leaves[:, j]
            order = np.argsort(column, kind='stable')
            unique, starts, sizes = np.unique(column[order], return_index=True, return_counts=True)
            self.lookup.append({int(k): order[start:start+size] for k, start, size in zip(unique, starts, sizes)})
        return self

    def weights(self, query):
        x = query[self.cols].to_numpy(float)
        if not np.isfinite(x).all():
            raise ValueError('Query has missing / infinite features')
        leaves = self.forest.apply(x)
        result = np.zeros((len(x), len(self.means)))
        for i in range(len(x)):
            for j, leaf in enumerate(leaves[i]):
                ids = self.lookup[j][int(leaf)]
                result[i, ids] += 1./(len(self.lookup)*len(ids))
        return result

    def predict(self, query):
        weights = self.weights(query)
        x = query[self.cols].to_numpy(float)
        outside = ((x < self.minimum) | (x > self.maximum)).any(axis=1)
        return [GaussianMixture(self.means, w, self.width, self.hosts) for w in weights], outside


class TwoStageDensity:
    """p(logM0|D)= integral p(logM0|logMz) p(logMz|D) dlogMz.

    Both factors and their smoothing are tuned using end-to-end validation
    NLL of logM0, on the same training folds as direct prediction.
    """
    def __init__(self, cols, leaf, ywidth, mwidth, cfg):
        self.first = ForestDensity(cols, leaf, mwidth, cfg, target='logMz')
        self.second = ForestDensity(['logMz'], leaf, ywidth, cfg)
        self.grid_size, self.mwidth = cfg.bridge_grid_size, float(mwidth)

    def fit(self, frame):
        self.first.fit(frame)
        self.second.fit(frame)
        edges = np.linspace(frame.logMz.min()-8*self.mwidth,
                            frame.logMz.max()+8*self.mwidth, self.grid_size+1)
        self.centers = (edges[:-1]+edges[1:])/2
        # Analytically integrate each first-stage Gaussian within each mass bin.
        cdf = ndtr((edges[:, None]-self.first.means[None, :])/self.mwidth)
        self.mass_bin_probability = np.diff(cdf, axis=0).T
        if not np.allclose(self.mass_bin_probability.sum(axis=1), 1., atol=1e-10):
            raise ValueError('Integration grid loses first-stage probability mass')
        self.transition = self.second.weights(pd.DataFrame({'logMz': self.centers}))
        self.bridge_outside = ((self.centers < frame.logMz.min()) | (self.centers > frame.logMz.max())).astype(float)
        return self

    def predict(self, query):
        weights = self.first.weights(query)
        mass_prob = weights@self.mass_bin_probability
        final = mass_prob@self.transition
        x = query[self.first.cols].to_numpy(float)
        outside = ((x < self.first.minimum) | (x > self.first.maximum)).any(axis=1)
        self.last_bridge_extrapolated_mass = mass_prob@self.bridge_outside
        return [GaussianMixture(self.second.means, w, self.second.width, self.second.hosts) for w in final], outside


def fit_density(name, cols, params, cfg, frame):
    if name.startswith('two_stage'):
        return TwoStageDensity(cols, **params, cfg=cfg).fit(frame)
    return ForestDensity(cols, **params, cfg=cfg).fit(frame)


def density_grid(name, cfg):
    values = product(cfg.leaf_grid, cfg.ywidth_grid, cfg.mwidth_grid if name.startswith('two_stage') else (None,))
    return [dict(leaf=int(leaf), ywidth=float(yw), **({'mwidth': float(mw)} if mw is not None else {}))
            for leaf, yw, mw in values]


def density_candidate_audit(splits, epochs, cfg):
    """One common candidate set; never skip an individual difficult fold."""
    if not epochs or not splits:
        raise ValueError('At least one eligible epoch and CV split is required')
    counts = [len(fit.loc[fit.snapshot.eq(sn)]) for fit, _ in splits for sn in epochs]
    minimum = min(counts)
    rows = []
    for name in feature_sets(cfg):
        for candidate, params in enumerate(density_grid(name, cfg)):
            eligible = minimum >= 2*params['leaf']
            rows.append(dict(model=name, candidate=candidate, minimum_cv_train=minimum,
                required_train=2*params['leaf'], eligible=eligible,
                reason='included' if eligible else 'too few rows for two leaves in the smallest CV training fold',
                **params))
    return pd.DataFrame(rows)


## 解析3・4：共通標本でのモデル比較

| モデル | 入力 | 意味 |
|---|---|---|
| `halo_oracle` | 真の $M_z$ | 観測では得られない基準 |
| `halo_plus_environment` | 真の $M_z$＋分光環境の銀河数 | Intermediate：halo massに対する環境の追加価値 |
| `core` | 推定恒星質量 | 現在の10に対応する出発点 |
| `photo_environment` | core＋測光環境の多半径銀河数 | 周辺の測光情報の価値 |
| `spec_narrow / spec_wide` | core＋分光環境、最大5 / 15 cMpc | 広い視野の情報の価値 |
| `deep_spec` | core＋深い選択での分光環境 | 深さと分光抽出率を増やす価値 |
| `geometry` | spec_wide＋質量和、近傍、形状 | 個数以外の空間・質量分布 |
| `population_proxy` | geometry＋SFR proxy | 恒星集団情報の探索的な追加 |
| `truth3d_reference` | 真の恒星質量＋3D環境 | 理想的な参照。性能の数学的上限とは限らない |
| `two_stage_population` | population_proxyと全く同じ観測入力 | 直接法と同条件の二段階control |

`halo_oracle`と直接法の違いを、同じ観測情報量での優劣とは解釈しない。
二段階controlは同じ標本で両factorを学び、WittenのSMHM relationを数値的に再現するものではない。
学習母集団・ラベルが同じcontrolを置くことで、halo massへ圧縮する近似を検証する。

主指標は未使用testのNLL、$M_0\geq10^{14}M_\odot$のBrier score、68/95%包含率。
区間幅だけが小さくなったことを改善としない。featureのtraining範囲外、または混合のhost ESS<5を
support不足として表示するが、全体の採点から除かない。これは簡易診断で、joint supportを保証しない。

平均は各時刻内で銀河を等重みとする。同じhostへ行く複数銀河は残し、同じ時刻を繰り返すと
独立標本数が増えるとは主張しない。差の区間はtestの最終hostを単位にしたpaired bootstrap。
host間の空間相関・宇宙分散・シミュレーション系統誤差はこの区間に含まれない。


In [ ]:
def tune_models(train, splits, epochs, cfg, progress=print, checkpoint=None):
    """No test access; equal fold and epoch weights, common validation caps."""
    candidate_audit = density_candidate_audit(splits, epochs, cfg)
    active = candidate_audit.loc[candidate_audit.eligible]
    for name in feature_sets(cfg):
        if not active.model.eq(name).any():
            raise ValueError('No candidate supports every CV training fold for '+name)
    allowed_keys = {(r.model, int(r.candidate), fold, int(sn))
                    for r in active.itertuples(index=False)
                    for fold in range(len(splits)) for sn in epochs}
    records, selected = [], {}
    if checkpoint is not None and Path(checkpoint).exists():
        saved = pd.read_csv(checkpoint, float_precision='round_trip')
        if saved.duplicated(['model', 'candidate', 'fold', 'snapshot']).any():
            raise ValueError('Duplicate CV checkpoint keys')
        records = saved.to_dict(orient='records')
    completed = {(r['model'], int(r['candidate']), int(r['fold']), int(r['snapshot'])) for r in records}
    if not completed.issubset(allowed_keys):
        raise ValueError('CV checkpoint contains ineligible candidates or different folds; use a new run directory')
    for row in records:
        expected = density_grid(row['model'], cfg)[int(row['candidate'])]
        if any(row.get(k) != v for k, v in expected.items()):
            raise ValueError('CV checkpoint parameters differ from the current candidate grid')
    for name, cols in feature_sets(cfg).items():
        for candidate, params in enumerate(density_grid(name, cfg)):
            if not ((active.model == name) & (active.candidate == candidate)).any():
                continue
            for fold, (all_fit, all_val) in enumerate(splits):
                for sn in epochs:
                    if (name, candidate, fold, int(sn)) in completed:
                        continue
                    fit = all_fit.loc[all_fit.snapshot.eq(sn)]
                    val = all_val.loc[all_val.snapshot.eq(sn)]
                    val = val.sample(min(len(val), cfg.cv_cap_per_snapshot), random_state=cfg.seed+fold+int(sn))
                    model = fit_density(name, cols, params, cfg, fit)
                    post, _ = model.predict(val)
                    records.append(dict(model=name, candidate=candidate, fold=fold, snapshot=int(sn),
                        N=len(val), nll=float(np.mean([-p.logpdf(y) for p, y in zip(post, val.logM0)])), **params))
                    if checkpoint is not None:
                        atomic_csv(pd.DataFrame(records), checkpoint)
        scores = pd.DataFrame([r for r in records if r['model'] == name]).groupby('candidate').nll.mean()
        best = int(scores.idxmin())
        selected[name] = dict(params=density_grid(name, cfg)[best], cv_nll=float(scores.loc[best]), features=cols)
        progress('CV complete: '+name+' '+str(selected[name]['params']))
    return selected, pd.DataFrame(records)


def score_predictions(posteriors, query, model_name, outside=None, bridge_mass=None):
    rows = []
    outside = np.zeros(len(query), bool) if outside is None else outside
    bridge_mass = np.full(len(query), np.nan) if bridge_mass is None else bridge_mass
    for p, row, out, bridge in zip(posteriors, query.itertuples(index=False), outside, bridge_mass):
        y = row.logM0
        pit = p.cdf(y)
        q16, med, q84 = [p.quantile(v) for v in (.16, .5, .84)]
        prob = 1-p.cdf(14.)
        diagnostics = p.diagnostics()
        rows.append(dict(model=model_name, snapshot=int(row.snapshot), redshift=float(row.redshift),
            SubhaloID=int(row.SubhaloID), GroupID_z0=int(row.GroupID_z0), logM0=float(y),
            median=med, q16=q16, q84=q84, W68_dex=q84-q16, pit=pit, nll=-p.logpdf(y),
            coverage68=.16 <= pit <= .84, coverage95=.025 <= pit <= .975,
            p_cluster=prob, brier=(prob-float(y >= 14))**2, outside_range=bool(out),
            support_ok=bool(not out and diagnostics['ESS_host'] >= 5),
            bridge_extrapolated_mass=float(bridge), **diagnostics))
    return pd.DataFrame(rows)


def evaluate_models(train, test, selected, epochs, cfg):
    """Fit frozen models; unsupported queries remain in the headline scores."""
    output, models = [], {}
    for sn in epochs:
        fit = train.loc[train.snapshot.eq(sn)]
        query = test.loc[test.snapshot.eq(sn)]
        if query.empty:
            raise ValueError('No test anchors at snapshot '+str(sn))
        query = query.sample(min(len(query), cfg.test_cap_per_snapshot), random_state=10000+cfg.seed+int(sn))
        for name, spec in selected.items():
            model = fit_density(name, spec['features'], spec['params'], cfg, fit)
            models[(int(sn), name)] = model
            post, outside = model.predict(query)
            bridge = model.last_bridge_extrapolated_mass if name.startswith('two_stage') else None
            output.append(score_predictions(post, query, name, outside, bridge))
        prior_width = selected['core']['params']['ywidth']
        prior = GaussianMixture(fit.logM0, np.ones(len(fit)), prior_width, fit.GroupID_z0)
        output.append(score_predictions([prior]*len(query), query, 'training_prior'))
    raw = pd.concat(output, ignore_index=True)
    summary = raw.groupby(['model', 'snapshot', 'redshift']).agg(
        N=('nll', 'size'), N_hosts=('GroupID_z0', 'nunique'), NLL=('nll', 'mean'),
        Brier=('brier', 'mean'), coverage68=('coverage68', 'mean'), coverage95=('coverage95', 'mean'),
        W68_dex=('W68_dex', 'mean'), supported_fraction=('support_ok', 'mean'),
        outside_fraction=('outside_range', 'mean'), ESS_host_median=('ESS_host', 'median'),
        bridge_extrapolated_mass=('bridge_extrapolated_mass', 'mean')).reset_index()
    return raw, summary, models


COMPARISONS = (
    ('halo_plus_environment', 'halo_oracle', 'environment_at_fixed_true_Mz'),
    ('photo_environment', 'core', 'photometric_environment'),
    ('spec_wide', 'photo_environment', 'spectroscopic_environment'),
    ('spec_wide', 'spec_narrow', 'wider_environment'),
    ('deep_spec', 'spec_wide', 'deeper_and_more_complete_spectroscopy'),
    ('geometry', 'spec_wide', 'mass_profile_and_geometry'),
    ('population_proxy', 'geometry', 'SFR_proxy_information'),
    ('population_proxy', 'two_stage_population', 'direct_vs_two_stage_same_data'),
)


def paired_differences(predictions, cfg):
    """Paired resampling of test z0 hosts; epochs receive equal weight.

    Inter-host correlations can remain inside the held-out slab, so intervals
    are descriptive host-bootstrap intervals, not cosmic-variance errors.
    """
    key = ['snapshot', 'SubhaloID', 'GroupID_z0']
    rng = np.random.default_rng(cfg.seed+908)
    rows = []
    for new, old, label in COMPARISONS:
        a = predictions.loc[predictions.model.eq(new)]
        b = predictions.loc[predictions.model.eq(old)]
        joined = a.merge(b, on=key, suffixes=('_new', '_old'), validate='one_to_one')
        if len(joined) != len(a) or len(joined) != len(b):
            raise ValueError('Models were not scored on identical test anchors')
        hosts = joined.GroupID_z0.unique()
        for metric in ('nll', 'brier', 'W68_dex'):
            tab = joined[key].copy()
            tab['delta'] = joined[metric+'_new']-joined[metric+'_old']
            estimate = tab.groupby('snapshot').delta.mean().mean()
            boot = []
            for _ in range(cfg.bootstrap_draws):
                multiplicity = pd.Series(rng.choice(hosts, len(hosts), replace=True)).value_counts()
                w = tab.GroupID_z0.map(multiplicity).fillna(0).to_numpy()
                vals = []
                for _, ids in tab.groupby('snapshot').groups.items():
                    ix = np.asarray(list(ids))
                    if w[ix].sum():
                        vals.append(np.average(tab.delta.to_numpy()[ix], weights=w[ix]))
                if len(vals) == tab.snapshot.nunique():
                    boot.append(np.mean(vals))
            if not boot:
                raise ValueError('No valid paired bootstrap replicates')
            lo, hi = np.quantile(boot, [.025, .975])
            rows.append(dict(comparison=label, new=new, reference=old, metric=metric,
                             delta=float(estimate), lo95=float(lo), hi95=float(hi),
                             N=len(tab), N_hosts=len(hosts), valid_draws=len(boot)))
    return pd.DataFrame(rows)


def calibration_table(predictions, bins=5):
    tab = predictions.copy()
    tab['probability_bin'] = np.minimum((tab.p_cluster*bins).astype(int), bins-1)
    tab['is_cluster'] = tab.logM0.ge(14).astype(float)
    return tab.groupby(['model', 'snapshot', 'probability_bin']).agg(
        N=('is_cluster', 'size'), mean_probability=('p_cluster', 'mean'),
        cluster_fraction=('is_cluster', 'mean')).reset_index()


## 解析5：一つの領域は、一つのdescendantになるか

**目的。** anchorの将来hostを、領域全体の将来と同一視できるかを調べる。
これは観測可能な特徴量ではなく、シミュレーションの正解側の監査である。

**手法。** labelを見る前に無作為抽出した30 anchorについて、真の位置の半径10 cMpc球を固定する。
親集団は保存下限以上の全銀河で、anchor自身も含む。個数重みと真の恒星質量重みを別々に集計し、
それぞれの $z=0$ hostに入る割合を求める。最も大きい割合を受け取るhostをdominantとする。
「最も大きいdescendant mass」を選ぶラベルは作らない。

未追跡銀河を分母から落とさず、dominant割合のlower/upper boundを示す。
未追跡の全重みが他のhostへ入っても順位が変わらない場合だけ、dominantの同定がstableと記録する。
この監査の真の3D領域は観測で定義した候補そのものではない。主解析のanchorラベルを
dominantラベルへ切り替えて比較するには、別に共通標本とsplitを再定義する必要がある。


In [ ]:
def region_audit(observed, anchors, meta, cfg, store):
    """Fixed truth 3D reference apertures; labels never used as predictors.

    Missing endpoints stay in denominators. Upper/lower bounds describe the
    unresolved allocation. No maximum-descendant-mass label is constructed.
    """
    if cfg.region_audit_n <= 0:
        return pd.DataFrame()
    chosen = anchors.sample(min(cfg.region_audit_n, len(anchors)), random_state=cfg.seed+201)
    tree = cKDTree(observed[POS].to_numpy(), boxsize=meta['boxsize_cMpc'])
    rows = []
    for anchor in chosen.itertuples(index=False):
        ids = tree.query_ball_point([getattr(anchor, c) for c in POS], cfg.region_radius_cMpc)
        members = observed.iloc[ids].copy()
        delta = periodic_delta(members[POS].to_numpy(), [getattr(anchor, c) for c in POS], meta['boxsize_cMpc'])
        members = members.loc[np.linalg.norm(delta, axis=1) < cfg.region_radius_cMpc]
        # The reference parent is all stored physical galaxies, including anchor.
        labels = [store.trace(int(anchor.snapshot), int(s)) for s in members.SubhaloID]
        label = pd.DataFrame(labels, index=members.index)
        known = label.TrackingStatus.eq('success') & np.isfinite(label.logM0) & label.GroupID_z0.ge(0)
        anchor_label = store.trace(int(anchor.snapshot), int(anchor.SubhaloID))
        for weighting in ('number', 'stellar_mass'):
            w = np.ones(len(members)) if weighting == 'number' else 10**members.logMstar.to_numpy()
            total = w.sum()
            grouped = pd.Series(w[known.to_numpy()], index=label.loc[known, 'GroupID_z0']).groupby(level=0).sum().sort_values(ascending=False)
            missing = w[~known.to_numpy()].sum()
            main = int(grouped.index[0]) if len(grouped) else -1
            largest = float(grouped.iloc[0]) if len(grouped) else 0.
            second = float(grouped.iloc[1]) if len(grouped) > 1 else 0.
            stable = bool(len(grouped) and largest > second+missing)
            anchor_known = anchor_label['TrackingStatus'] == 'success'
            anchor_weight = float(grouped.get(anchor_label['GroupID_z0'], 0)) if anchor_known else 0.
            # If the anchor endpoint is unknown, its host may be any of the
            # known hosts, not only an entirely untracked host.
            anchor_upper = (anchor_weight+missing)/total if anchor_known else (largest+missing)/total
            rows.append(dict(snapshot=int(anchor.snapshot), SubhaloID=int(anchor.SubhaloID), weighting=weighting,
                N_members=len(members), tracking_weight_fraction=float(1-missing/total),
                known_descendant_hosts=len(grouped), dominant_known_host=main, dominant_identity_stable=stable,
                f_main_lower=largest/total, f_main_upper=(largest+missing)/total,
                f_anchor_lower=anchor_weight/total, f_anchor_upper=anchor_upper,
                anchor_is_dominant_known=bool(len(grouped) and anchor_label['TrackingStatus'] == 'success'
                                               and main == anchor_label['GroupID_z0'])))
    return pd.DataFrame(rows)


## 実装の確認用データと、共通標本の組立

次の人工データ関数は `MODE='synthetic_demo'` でのみ呼ばれる。
TNG modeでは人工データや人工の最終hostを使用しない。


In [ ]:
class MemoryStore:
    """Endpoint lookup for deliberately synthetic software checks only."""
    def __init__(self, labels):
        self.labels = labels
        self.fingerprint = {'mode': 'synthetic_demo', 'version': VERSION}

    def trace(self, snapshot, sid):
        return dict(self.labels[(int(snapshot), int(sid))])

    def close(self):
        pass


def synthetic_inputs(cfg, n_groups=150, per_group=25, boxsize=300.):
    """Artificial clustering and labels; never a substitute for TNG results."""
    rng = np.random.default_rng(cfg.seed+710)
    centers = rng.uniform(0, boxsize, size=(n_groups, 3))
    group_mass = rng.uniform(12., 15., size=n_groups)
    raw, metadata, labels = {}, {}, {}
    for snapshot in cfg.snapshots:
        n = n_groups*per_group
        gids = np.repeat(np.arange(n_groups), per_group)
        pos = (centers[gids]+rng.normal(0, 3, (n, 3))) % boxsize
        logmass = np.maximum(cfg.archive_logmstar_min, rng.normal(8.9+.15*(group_mass[gids]-13.5), .45))
        sfr = 10**(logmass-8.2+rng.normal(0, .3, n))
        z, hz = 8.01217294886593, 1018.
        frame = pd.DataFrame(dict(SubhaloID=np.arange(n), snapshot=int(snapshot), redshift=z,
            GroupID=np.arange(n), logMstar=logmass, SFR=sfr, Nstar=np.maximum(1, (10**(logmass-7.1)).astype(int)),
            logMz=11+.3*(logmass-9)+rng.normal(0, .2, n), is_central=np.ones(n, bool)))
        for j, c in enumerate(POS):
            frame[c] = pos[:, j]
        for c in VEL:
            frame[c] = rng.normal(0, 180, n)
        for sid, host in enumerate(gids):
            labels[(int(snapshot), sid)] = dict(TrackingStatus='success', SubhaloID_z0=int(host),
                TerminalSnap=99, GroupID_z0=int(host), logM0=float(group_mass[host]))
        raw[int(snapshot)] = frame
        metadata[int(snapshot)] = dict(simulation='SYNTHETIC SOFTWARE DEMO', snapshot=int(snapshot),
            redshift=z, H_kms_Mpc=hz, boxsize_cMpc=boxsize, h=.6774,
            archive_logmstar_min=cfg.archive_logmstar_min,
            synthetic_note='Artificial coordinates, galaxies, and descendant labels; no scientific result')
    return raw, metadata, MemoryStore(labels)


def build_experiment(raw_catalogs, metadata, cfg, store, progress=print):
    """Freeze observed anchors before looking up endpoints; retain audits."""
    frames, quality, observed_by_snap, all_anchors = [], [], {}, {}
    for sn in cfg.snapshots:
        meta = metadata[sn]
        cfg.validate(meta['boxsize_cMpc'])
        observed = mock_observe(raw_catalogs[sn], meta, cfg)
        anchors = select_anchors(observed, cfg)
        labeled = attach_labels(anchors, store)
        tracked = labeled.TrackingStatus.eq('success') & np.isfinite(labeled.logM0) & labeled.GroupID_z0.ge(0)
        valid = tracked & np.isfinite(labeled.logMz)
        # Diagnostic bins include successes AND failures; no completeness correction.
        labeled['mass_bin'] = np.floor(labeled.logMstar_obs*2)/2
        labeled['tracked'] = tracked
        labeled['valid_label'] = valid
        report = labeled.groupby('mass_bin').agg(N_selected=('SubhaloID', 'size'),
            N_tracked=('tracked', 'sum'), tracking_fraction=('tracked', 'mean'),
            N_valid=('valid_label', 'sum'), model_fraction=('valid_label', 'mean'),
            median_Nstar=('Nstar', 'median')).reset_index()
        report['snapshot'], report['redshift'] = sn, meta['redshift']
        quality.append(report)
        observed_by_snap[sn], all_anchors[sn] = observed, labeled
        for scenario in ('truth3d', 'photo', 'spec', 'deep_spec'):
            features = environment_features(observed, anchors, meta, cfg, scenario)
            labeled = labeled.merge(features, on=['snapshot', 'SubhaloID'], validate='one_to_one')
        labeled = labeled.loc[valid.to_numpy()].copy()
        if labeled.empty:
            raise ValueError('No successfully tracked common anchors')
        frames.append(labeled)
        progress('Features complete: snapshot '+str(sn)+'; common tracked anchors='+str(len(labeled)))
    data = pd.concat(frames, ignore_index=True)
    if data.duplicated(['snapshot', 'SubhaloID']).any():
        raise ValueError('Duplicate anchors')
    # Each final host must have exactly one final M200c at every source epoch.
    if (data.groupby('GroupID_z0').logM0.agg(lambda x: x.max()-x.min()) > 1e-8).any():
        raise ValueError('Inconsistent descendant mass for the same z0 FoF')
    for cols in feature_sets(cfg).values():
        if not np.isfinite(data[cols].to_numpy(float)).all():
            raise ValueError('A model input is missing or infinite')
    return data, pd.concat(quality, ignore_index=True), observed_by_snap, all_anchors


def experiment_manifest(cfg, metadata, store, mode, implementation_sha256):
    return dict(version=VERSION, implementation_sha256=implementation_sha256,
                mode=mode, config=asdict(cfg), inputs=metadata, traces=store.fingerprint,
                target='anchor descendant z0 FoF M200c; galaxy-weighted per epoch',
                direct_inputs='mock noisy stellar mass and mock environmental/population summaries',
                selection='observed shallow detection and logMstar_obs cut, before descendant lookup',
                limitations=['single-snapshot periodic mock, not a lightcone',
                    'Gaussian errors and mass-dependent selection are uncalibrated design assumptions',
                    'SFR feature is a noisy instantaneous-SFR proxy, not a measured Balmer/UV SFR',
                    'no UV, line-flux, dust, metallicity, ALMA, Lyalpha or radiative-transfer likelihood',
                    'posterior is conditional on successful tracing and finite halo mass',
                    'two-stage control is learned on the same sample; not a numerical reproduction of Witten SMHM',
                    'host-bootstrap intervals omit inter-host spatial covariance and cosmic variance'])


def analytic_self_checks():
    """Cheap deterministic checks suitable for every TNG kernel startup."""
    assert np.isclose(periodic_delta(.2, 9.8, 10.), .4)
    assert wrapped_window_probability(np.array([0., 2.]), 0., 1., 10.).tolist() == [1., 0.]
    grid = np.linspace(-5, 5, 2001)
    probability = wrapped_window_probability(grid, 2., 1., 10.)
    assert np.all((probability >= 0) & (probability <= 1))
    assert np.isclose(np.trapz(probability, grid), 2., atol=1e-6)
    mixture = GaussianMixture([13., 14.], [1., 2.], .2)
    assert np.isclose(mixture.cdf(mixture.quantile(.5)), .5, atol=1e-8)
    assert np.isclose(mixture.cdf(20)-mixture.cdf(0), 1.)
    return 'Passed: periodic boundary, velocity weights, and normalized probability / quantile checks'


## 実行1：入力の読み込みと自己検査

最初に周期境界・membership確率・混合分布の正規化を確認する。
TNG modeでは、読んだheaderとDM粒子数からTNG300-1であることも検査する。
入力の読み込みに失敗した場合は停止し、demoへ自動的に切り替えない。


In [ ]:
print(analytic_self_checks())
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
if DEMO:
    raw_catalogs, metadata, store = synthetic_inputs(CFG, n_groups=180, per_group=20)
    make_store = lambda: MemoryStore(store.labels)
else:
    BASE = discover_base(BASE_PATH)
    print('Reading TNG300-1:', BASE)
    raw_catalogs, metadata = {}, {}
    for sn in CFG.snapshots:
        raw_catalogs[sn], metadata[sn] = load_tng_snapshot(BASE, sn, CFG, CACHE_ROOT/'inputs')
    make_store = lambda: DescendantStore(BASE, CACHE_ROOT/'inputs', legacy_csv=LEGACY_CSV)
    store = make_store()

manifest = experiment_manifest(CFG, metadata, store, MODE, IMPLEMENTATION_SHA256)
manifest['packages'] = dict(python=platform.python_version(), numpy=np.__version__,
                           pandas=pd.__version__, scipy=scipy.__version__, sklearn=sklearn.__version__)
RUN_DIR = CACHE_ROOT/'runs'/MODE/signature(manifest)[:16]
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR/'manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
display(pd.DataFrame([dict(snapshot=sn, redshift=metadata[sn]['redshift'], N_tracers=len(raw_catalogs[sn]),
                           box_cMpc=metadata[sn]['boxsize_cMpc']) for sn in CFG.snapshots]))
print(LABEL_PREFIX+'run directory:', RUN_DIR)

def save_figure(fig, name):
    fig.suptitle(LABEL_PREFIX+name.replace('_', ' '), fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, .94))
    fig.savefig(RUN_DIR/(name+'.png'), dpi=160, bbox_inches='tight')
    plt.show()


## 実行2：共通anchor、追跡完全性、多半径環境の構築

各観測条件で同じanchorを用い、feature計算後に有効な共通ラベルを持つ行で推論標本を作る。
以下の表で、選択数・有効ラベル数・星粒子数を確認する。
推論は追跡成功と有限な現在host質量に条件づけられ、未追跡の母集団への補正はしない。
`Nstar`は保存されたPartType4要素数で、windを除いたformed-star数とは限らず、収束の証明には使わない。
同じrunではfeature CSVを再利用する。別の誤差seedや深さは別runになる。


In [ ]:
try:
    if (RUN_DIR/'features.csv').exists() and (RUN_DIR/'tracking_quality.csv').exists():
        data = pd.read_csv(RUN_DIR/'features.csv', float_precision='round_trip')
        tracking_quality = pd.read_csv(RUN_DIR/'tracking_quality.csv')
        observed_by_snap = {sn: mock_observe(raw_catalogs[sn], metadata[sn], CFG) for sn in CFG.snapshots}
        parent_anchors = {sn: select_anchors(observed_by_snap[sn], CFG) for sn in CFG.snapshots}
        print('Reused features from this fingerprinted run')
    else:
        data, tracking_quality, observed_by_snap, parent_anchors = build_experiment(
            raw_catalogs, metadata, CFG, store)
        atomic_csv(data, RUN_DIR/'features.csv')
        atomic_csv(tracking_quality, RUN_DIR/'tracking_quality.csv')
finally:
    store.close()
if data.duplicated(['snapshot', 'SubhaloID']).any():
    raise ValueError('Invalid feature-cache keys')
display(tracking_quality)


## 実行3：分割とCVの成立条件の確認

spatial/host purgeのため、多くのanchorが学習・testの両方から外れることがある。
その数を隠さず保存する。`cv_fold_audit.csv`には各条件の必要数と不成立の理由も記録する。
CV不成立なら停止し、testを使って分割やパラメータを調整しない。
対象時刻・計算上限を変える再設計は、新しいrunとして扱う。


In [ ]:
boxes = [metadata[sn]['boxsize_cMpc'] for sn in CFG.snapshots]
if not np.allclose(boxes, boxes[0]):
    raise ValueError('All epochs must use the same periodic box')
partitioned, split_spec = spatial_split(data, metadata[CFG.snapshots[0]], CFG)
train = partitioned.loc[partitioned.partition.eq('train')].copy()
test = partitioned.loc[partitioned.partition.eq('test')].copy()
folds, cv_audit, EPOCHS = cv_splits(train, split_spec, CFG)
atomic_csv(partitioned[['snapshot', 'SubhaloID', 'GroupID_z0', 'split_coordinate', 'partition', 'cv_fold']],
           RUN_DIR/'split.csv')
atomic_csv(cv_audit, RUN_DIR/'cv_fold_audit.csv')
(RUN_DIR/'split_spec.json').write_text(json.dumps(split_spec, indent=2))
display(partitioned.groupby(['snapshot', 'partition']).size().rename('N').reset_index())
display(cv_audit)
if not EPOCHS:
    details = cv_audit.loc[~cv_audit.eligible, ['snapshot', 'fold', 'reason']].to_string(index=False)
    raise ValueError('No epoch supports every CV fold. Inspect cv_fold_audit.csv; '
                     'the spatial/host separation has been retained.\n'+details)
if set(EPOCHS) != set(CFG.snapshots):
    print('Excluded from ALL model comparisons:', sorted(set(CFG.snapshots)-set(EPOCHS)))
print('Epochs used by ALL models:', EPOCHS)


## 実行4：trainingで環境の回収を可視化する

各半径について、真の3D球内銀河数と、測光・分光・深い分光環境での有効銀河数の中央値を比較する。
3Dのtracerは真の質量閾値、観測tracerは推定質量と検出で選ぶため、比は単なる測定biasではない。
異なる奥行き・閾値・分光率による違いを含む。ここではtrainingだけを描き、半径は事前固定のまま。


In [ ]:
recovery_rows = []
for sn in EPOCHS:
    frame = train.loc[train.snapshot.eq(sn)]
    for scenario in ('truth3d', 'photo', 'spec', 'deep_spec'):
        for radius in CFG.radii_cMpc:
            values = np.expm1(frame[scenario+'__lnN_R'+rtag(radius)])
            q16, median, q84 = np.quantile(values, [.16, .5, .84])
            recovery_rows.append(dict(snapshot=sn, scenario=scenario, radius_cMpc=radius,
                                      q16=q16, median=median, q84=q84))
recovery = pd.DataFrame(recovery_rows)
atomic_csv(recovery, RUN_DIR/'environment_recovery_training.csv')
fig, axes = plt.subplots(1, len(EPOCHS), figsize=(6*len(EPOCHS), 4), squeeze=False)
for ax, sn in zip(axes[0], EPOCHS):
    for scenario, g in recovery.loc[recovery.snapshot.eq(sn)].groupby('scenario', sort=False):
        ax.plot(g.radius_cMpc, g['median'], marker='o', label=scenario)
    ax.set(xlabel='Radius [cMpc]', ylabel='Median neighbor count / effective count', title='snapshot '+str(sn))
    ax.legend()
save_figure(fig, 'environment_recovery_training')


## 実行5：training CVでモデルを固定する

全モデルで同じvalidation銀河を使用する。計算量を抑えるcapもモデル間で共有する。
`cv_candidate_audit.csv`で、最小foldの学習数と候補ごとの必要数・採用可否を確認する。
候補の採用可否にはvalidation/testの質量や予測損失を用いない。
CVの1モデル・1候補・1fold・1時刻ごとにCSVへ保存し、通常の中断後には完了分を再利用する。
`selected_models.json`はtestの評価前に確定する。


In [ ]:
candidate_audit = density_candidate_audit(folds, EPOCHS, CFG)
atomic_csv(candidate_audit, RUN_DIR/'cv_candidate_audit.csv')
display(candidate_audit.groupby(['leaf', 'minimum_cv_train', 'required_train', 'eligible']).size().rename('N_candidates').reset_index())
selected, cv_scores = tune_models(train, folds, EPOCHS, CFG, checkpoint=RUN_DIR/'cv_scores.csv')
(RUN_DIR/'selected_models.json').write_text(json.dumps(selected, indent=2, ensure_ascii=False))
display(pd.DataFrame([dict(model=k, cv_nll=v['cv_nll'], n_features=len(v['features']), **v['params'])
                      for k, v in selected.items()]))


## 実行6：未使用testで分布を評価する

NLLは $-\log p(y_{true}\mid D)$、Brierは $(P[M_0\geq10^{14}]-\mathbf1[M_0\geq10^{14}])^2$。
PITから中心68/95%包含率を求める。分布の狭さ、包含率、support率を一緒に読む。
サポート不足や外挿queryも採点に含め、良い点だけを選ばない。


In [ ]:
predictions, validation, fitted_models = evaluate_models(train, test, selected, EPOCHS, CFG)
atomic_csv(predictions, RUN_DIR/'test_predictions.csv')
atomic_csv(validation, RUN_DIR/'test_validation.csv')
differences = paired_differences(predictions, CFG)
atomic_csv(differences, RUN_DIR/'paired_differences.csv')
calibration = calibration_table(predictions)
atomic_csv(calibration, RUN_DIR/'calibration_bins.csv')
display(validation)
display(differences.loc[differences.metric.eq('nll')])


## 実行7：改善量、確率calibration、予測区間を読む

paired差は「新しい情報を加えたモデル − 比較元」。NLL/Brierは負なら改善する。
区間幅の差が負でも、包含率が悪化していれば改善とはしない。
同じtestを何度も見てモデルを選び直した場合、このtestは未使用評価ではなくなる。


In [ ]:
delta = differences.loc[differences.metric.eq('nll')].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(9, 5))
ypos = np.arange(len(delta))
for i, row in delta.iterrows():
    ax.plot([row.lo95, row.hi95], [i, i], color='tab:blue')
    ax.plot(row.delta, i, 'o', color='tab:blue')
ax.axvline(0, color='black', ls='--', lw=1)
ax.set_yticks(ypos)
ax.set_yticklabels(delta.comparison)
ax.set_xlabel('Delta NLL: new minus reference (host bootstrap interval)')
save_figure(fig, 'paired_information_gain')

chosen_models = ['core', 'spec_wide', 'population_proxy', 'two_stage_population']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name in chosen_models:
    p = predictions.loc[predictions.model.eq(name)]
    axes[0].hist(p.pit, bins=np.linspace(0, 1, 11), density=True, histtype='step', label=name)
    # Bins are shown per epoch; do not pool incompatible epoch calibrations.
    for sn in EPOCHS:
        g = calibration.loc[calibration.model.eq(name) & calibration.snapshot.eq(sn)]
        axes[1].plot(g.mean_probability, g.cluster_fraction, 'o-', label=name+' sn='+str(sn))
    g = validation.loc[validation.model.eq(name)]
    axes[2].scatter(g.W68_dex, g.coverage68, label=name)
axes[0].axhline(1, color='gray', ls='--')
axes[0].set(xlabel='PIT', ylabel='Density')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set(xlabel='Predicted P(cluster)', ylabel='Observed cluster fraction', xlim=(0, 1), ylim=(0, 1))
axes[2].axhline(.68, color='gray', ls='--')
axes[2].set(xlabel='Mean 68% interval width [dex]', ylabel='68% coverage', ylim=(0, 1))
axes[0].legend(fontsize=7)
axes[1].legend(fontsize=6)
axes[2].legend(fontsize=7)
save_figure(fig, 'probability_calibration')


## 実行8：真の $M_z$ に条件づけた比較の診断

halo mass baselineへの環境の追加価値は、`halo_plus_environment − halo_oracle`の未使用NLLで判定する。
以下は真の $M_z$ の0.5 dex binに分けた診断。各binの天体数も示す。
このbin分割を使ってtestに最適な閾値を選ぶことはしない。
改善は、この母集団・観測条件・推定器の範囲での予測的な追加情報を示す。
有限標本の結果だけから、数学的な条件付き独立性の一般的な証明・反証とはしない。


In [ ]:
halo_test = predictions.loc[predictions.model.isin(['halo_oracle', 'halo_plus_environment'])].merge(
    test[['snapshot', 'SubhaloID', 'logMz']], on=['snapshot', 'SubhaloID'], validate='many_to_one')
halo_test['logMz_bin'] = np.floor(halo_test.logMz*2)/2
halo_bins = halo_test.groupby(['snapshot', 'model', 'logMz_bin']).agg(
    N=('nll', 'size'), N_hosts=('GroupID_z0', 'nunique'), NLL=('nll', 'mean'), Brier=('brier', 'mean')).reset_index()
atomic_csv(halo_bins, RUN_DIR/'halo_conditional_bins.csv')
display(halo_bins)


## 実行9：同じtest銀河の直接法・二段階法のPDF

最初の採点対象銀河を機械的に選び、真の結果に合わせて見栄えの良い例を探さない。
二段階法は中間halo massの分布を積分している。線は有限シミュレーションの条件付き分布で、
観測装置・銀河形成モデル・宇宙論の系統誤差まで含む事後分布ではない。


In [ ]:
example = predictions.loc[predictions.model.eq('population_proxy')].iloc[0]
query = test.loc[test.snapshot.eq(example.snapshot) & test.SubhaloID.eq(example.SubhaloID)].iloc[[0]]
grid = np.linspace(min(train.logM0.min(), query.logM0.iloc[0])-1,
                   max(train.logM0.max(), query.logM0.iloc[0])+1, 400)
fig, ax = plt.subplots(figsize=(8, 4))
example_rows = []
for name in ('core', 'spec_wide', 'population_proxy', 'two_stage_population'):
    model = fitted_models[(int(example.snapshot), name)]
    post, outside = model.predict(query)
    p = post[0]
    ax.plot(grid, np.exp(p.logpdf(grid)), label=name)
    example_rows.append(dict(model=name, outside_range=bool(outside[0]),
        median=p.quantile(.5), q16=p.quantile(.16), q84=p.quantile(.84),
        P_cluster=1-p.cdf(14), **p.diagnostics()))
ax.axvline(query.logM0.iloc[0], color='black', ls='--', label='TNG label' if not DEMO else 'Synthetic label')
ax.set(xlabel='log10(M0 / Msun)', ylabel='Conditional density [dex^-1]')
ax.legend(fontsize=8)
save_figure(fig, 'example_conditional_densities')
example_table = pd.DataFrame(example_rows)
atomic_csv(example_table, RUN_DIR/'example_posterior_summary.csv')
display(example_table)


## 実行10：領域の行先とanchorの代表性

子孫追跡を領域内のtracerにも拡張するため、追加I/Oが必要になる。
最初は30 anchorで監査する。`region_audit_n=0`にすれば明示的に省略できる。
追跡済みの天体は同じSQLiteから再利用し、主モデルの入力や採点にはこの表を用いない。


In [ ]:
region_tables = []
if CFG.region_audit_n > 0:
    audit_store = make_store()
    try:
        for sn in EPOCHS:
            region_tables.append(region_audit(observed_by_snap[sn], parent_anchors[sn],
                                              metadata[sn], CFG, audit_store))
    finally:
        audit_store.close()
if region_tables:
    region_fates = pd.concat(region_tables, ignore_index=True)
    atomic_csv(region_fates, RUN_DIR/'region_descendant_audit.csv')
    display(region_fates)
    fig, ax = plt.subplots(figsize=(7, 4))
    for weighting, g in region_fates.groupby('weighting'):
        ax.scatter(g.f_main_lower, g.f_anchor_lower, label=weighting, alpha=.6)
    ax.plot([0, 1], [0, 1], 'k--')
    ax.set(xlabel='Dominant host fraction: lower bound', ylabel='Anchor host fraction: lower bound',
           xlim=(0, 1), ylim=(0, 1))
    ax.legend()
    save_figure(fig, 'region_descendant_allocation')
else:
    print('Region audit explicitly disabled')


## 結果の保存と、次の観測モデルへ進める際の判断

`test_validation.csv`と`paired_differences.csv`で、広い視野、分光、深い選択、形状、SFR proxyの追加を比べる。
優先順位は、NLL/Brierが改善し、包含率・supportを維持するかで判断する。
同じtestで最善の追観測設定を選んだ後の性能を確定するには、別の未使用領域が必要になる。

この段階で実装していない、実測のfilter flux・line EW選択、dust、金属量、ALMA、Lyαは、
実データまたは妥当性を確認したsynthetic observable catalogとselection functionを用意してから追加する。
真の物理量をそのまま観測特徴量と呼んで置き換えない。
観測時間あたりの情報量の比較には、装置の感度・pointing・target配置のモデルも別途必要である。

保存される主なファイル：`manifest.json`、`tracking_quality.csv`、`features.csv`、`split.csv`、
`cv_fold_audit.csv`、`cv_candidate_audit.csv`、`cv_scores.csv`、`selected_models.json`、`test_predictions.csv`、
`test_validation.csv`、`paired_differences.csv`、`calibration_bins.csv`、各PNG。
region監査を有効にすると`region_descendant_audit.csv`も保存する。
モデルそのもののpickleは保存しない。固定した設定とcatalogから再学習する。

誤差・深さ・seed・半径・モデルgridを変えた場合は、新しいrunになる。
古いrunと比較する際は、標本・選択・ラベル・splitが一致するかmanifestを先に確認する。


In [ ]:
completion = dict(mode=MODE, scientific_TNG_run=not DEMO, completed_epochs=[int(s) for s in EPOCHS],
                  n_test_anchors=int(predictions.loc[predictions.model.eq('core')].shape[0]),
                  result_directory=str(RUN_DIR), implementation_sha256=IMPLEMENTATION_SHA256)
(RUN_DIR/'completion.json').write_text(json.dumps(completion, indent=2))
print(LABEL_PREFIX+'completed')
print('Results:', RUN_DIR)
if DEMO:
    print('Artificial software-verification results only. Run MODE=tng for scientific analysis.')
